# Patches — Bloc 4b, Sessió 7
### `pretty_midi`, arpegiador i timing precís

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Per guardar els teus canvis: `File → Save a copy in Drive`.

El sintetitzador en temps real (`MidiSynth`) es prova a **Thonny**. Aquí usem `MidiSynthRender` (renderitzat a array) i la simulació de timing — ambdós funcionen perfectament a Colab sense necessitar hardware d'àudio.

In [ ]:
!apt-get install -y fluidsynth fluid-soundfont-gm -q > /dev/null
!pip install pyfluidsynth pretty_midi mido -q

import pretty_midi
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import time
import urllib.request

base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/04_midi/sessio07_sequenciador/recursos/"
urllib.request.urlretrieve(base_url + "midisynth.py", "midisynth.py")

import sys
sys.path.insert(0, '.')
from midisynth import MidiSynthRender

print("Tot carregat!")

## 1. Crear notes amb pretty_midi

In [ ]:
pm = pretty_midi.PrettyMIDI(initial_tempo=120)
instrument = pretty_midi.Instrument(program=0)  # Acoustic Grand Piano

pattern = [60, 64, 67, 72]
note_duration = 0.25

t = 0.0
for pitch in pattern:
    nota = pretty_midi.Note(velocity=90, pitch=pitch, start=t, end=t+note_duration)
    instrument.notes.append(nota)
    t += note_duration

pm.instruments.append(instrument)
pm.write('sequencia.mid')

audio = pm.fluidsynth(fs=44100)
Audio(audio, rate=44100)

## 2. L'arpegiador

In [ ]:
def arpegiador(acord, n_repeticions, mode='up'):
    if mode == 'up':
        seq = acord
    elif mode == 'down':
        seq = acord[::-1]
    elif mode == 'updown':
        seq = acord + acord[::-1][1:-1]
    return seq * n_repeticions

def crea_midi(notes, note_duration, tempo=120):
    pm = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    instrument = pretty_midi.Instrument(program=0)
    t = 0.0
    for pitch in notes:
        nota = pretty_midi.Note(velocity=85, pitch=pitch, start=t, end=t+note_duration*0.9)
        instrument.notes.append(nota)
        t += note_duration
    pm.instruments.append(instrument)
    return pm

# Provem els 3 modes
acord = [60, 64, 67]

for mode in ['up', 'down', 'updown']:
    notes = arpegiador(acord, n_repeticions=2, mode=mode)
    pm_arp = crea_midi(notes, note_duration=0.15)
    audio_arp = pm_arp.fluidsynth(fs=44100)
    print(f"Mode: {mode} — {notes}")
    display(Audio(audio_arp, rate=44100))

## 3. El problema del timing: simulació de timestamps

Simulem 30 notes a tempo ràpid (140bpm, corxeres) amb les dues estratègies, i mesurem el desfasament entre el temps esperat i el temps real d'execució.

In [ ]:
n_notes = 30
tempo_bpm = 140
durada_corxera = 60 / tempo_bpm / 2  # segons

# --- Estrategia A: time.sleep() repetit ---
timestamps_esperats_A = []
timestamps_reals_A = []

t0 = time.perf_counter()
t_esperat = 0.0
for i in range(n_notes):
    timestamps_esperats_A.append(t_esperat)
    timestamps_reals_A.append(time.perf_counter() - t0)
    time.sleep(durada_corxera)
    t_esperat += durada_corxera

# --- Estrategia B: temps objectiu absolut ---
timestamps_esperats_B = []
timestamps_reals_B = []

t0 = time.perf_counter()
t_acumulat = 0.0
for i in range(n_notes):
    t_objectiu = t0 + t_acumulat
    while time.perf_counter() < t_objectiu:
        pass
    timestamps_esperats_B.append(t_acumulat)
    timestamps_reals_B.append(time.perf_counter() - t0)
    t_acumulat += durada_corxera

print("Simulació completada")

In [ ]:
# Calculem la deriva (diferencia entre esperat i real) per a cada nota
deriva_A = np.array(timestamps_reals_A) - np.array(timestamps_esperats_A)
deriva_B = np.array(timestamps_reals_B) - np.array(timestamps_esperats_B)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(deriva_A * 1000, 'o-', color='crimson', label='Estratègia A (sleep)')
ax1.plot(deriva_B * 1000, 'o-', color='seagreen', label='Estratègia B (perf_counter)')
ax1.set_xlabel('Número de nota')
ax1.set_ylabel('Deriva (ms)')
ax1.set_title('Deriva acumulada: temps real - temps esperat')
ax1.legend()
ax1.axhline(0, color='gray', linewidth=0.5)

ax2.bar(['Estratègia A', 'Estratègia B'],
        [abs(deriva_A[-1])*1000, abs(deriva_B[-1])*1000],
        color=['crimson', 'seagreen'])
ax2.set_ylabel('Deriva final (ms)')
ax2.set_title(f'Deriva total després de {n_notes} notes')

plt.tight_layout()
plt.show()

print(f"Deriva final Estratègia A: {deriva_A[-1]*1000:.1f} ms")
print(f"Deriva final Estratègia B: {deriva_B[-1]*1000:.1f} ms")
print(f"\nA {tempo_bpm}bpm, una corxera dura {durada_corxera*1000:.1f} ms.")
print(f"Una deriva de {abs(deriva_A[-1])*1000:.0f}ms és clarament audible en un patró rítmic.")

## 4. Escoltant la diferència: renderitzem amb cada timing

Apliquem els timestamps de cada estratègia a notes reals i escoltem el resultat.

In [ ]:
def crea_midi_amb_timestamps(timestamps, pitch=60, dur=0.08):
    pm = pretty_midi.PrettyMIDI()
    instrument = pretty_midi.Instrument(program=118)  # Synth drum
    for t in timestamps:
        nota = pretty_midi.Note(velocity=100, pitch=pitch, start=t, end=t+dur)
        instrument.notes.append(nota)
    pm.instruments.append(instrument)
    return pm

print("Patró amb timestamps IDEALS (com hauria de sonar):")
pm_ideal = crea_midi_amb_timestamps(timestamps_esperats_B)
display(Audio(pm_ideal.fluidsynth(fs=44100), rate=44100))

print("\nPatró amb timestamps REALS de l'Estratègia A (sleep) — nota la deriva:")
pm_a = crea_midi_amb_timestamps(timestamps_reals_A)
display(Audio(pm_a.fluidsynth(fs=44100), rate=44100))

print("\nPatró amb timestamps REALS de l'Estratègia B (perf_counter) — molt més fidel:")
pm_b = crea_midi_amb_timestamps(timestamps_reals_B)
display(Audio(pm_b.fluidsynth(fs=44100), rate=44100))

## 5. MidiSynthRender — síntesi nota a nota (com a Thonny, pero en array)

In [ ]:
synth = MidiSynthRender()

trossos = []
for pitch in [60, 64, 67, 72]:
    synth.note_on(pitch, velocity=90)
    trossos.append(synth.render(0.2))
    synth.note_off(pitch)
    trossos.append(synth.render(0.05))

resultat = np.concatenate(trossos)
synth.close()

Audio(resultat, rate=44100)